## 介紹 

本課程將涵蓋： 
- 甚麼是函數調用及其應用場景 
- 如何使用 OpenAI 建立函數調用 
- 如何將函數調用整合到應用程式中 

## 學習目標 

完成本課程後，您將會知道如何並理解： 

- 使用函數調用的目的 
- 使用 OpenAI 服務設置函數調用 
- 為您的應用程式設計有效的函數調用 


## 理解函數呼叫 

在這課中，我們想為我們的教育創業公司建立一個功能，讓用戶可以使用聊天機械人找到技術課程。我們會根據用戶的技能水平、當前角色與感興趣的技術來推薦課程。 

為完成這項任務，我們將結合使用： 
 - `OpenAI` 為用戶創建聊天體驗
 - `Microsoft Learn Catalog API` 根據用戶要求協助找到課程
 - `函數呼叫` 將用戶查詢發送給函數以進行 API 請求。 

開始前，讓我們先看看為何我們會想使用函數呼叫： 


### 為何要使用函式調用

如果你已完成本課程的其他任何課程，你大概已經了解使用大型語言模型（LLM）的強大功能。希望你也能看到它們的一些限制。

函式調用是 OpenAI 服務的一項功能，旨在解決以下挑戰：

回應格式不一致：
- 在函式調用出現之前，大型語言模型的回應是無結構且不一致的。開發者必須編寫複雜的驗證程式碼來處理輸出中的各種變化。

與外部資料整合有限：
- 在此功能出現之前，將應用程式其他部分的資料整合到聊天上下文中是困難的。

透過標準化回應格式並實現與外部資料的無縫整合，函式調用簡化了開發，減少了額外驗證邏輯的需求。

使用者無法取得像「斯德哥爾摩現在的天氣如何？」這樣的答案。這是因為模型只能使用訓練資料的時間範圍內的資訊。

讓我們來看以下範例說明這個問題：

假設我們想建立一個學生資料庫，這樣才能為他們建議合適的課程。下面有兩個學生的描述，它們所包含的資料非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想把這個發送給大型語言模型 (LLM) 來解析數據。之後可以用於我們的應用程式，將其傳送到 API 或存入資料庫。

讓我們創建兩個相同的提示，用來指示 LLM 我們感興趣的信息：


我們想將此發送給大型語言模型（LLM），以解析對我們產品重要的部分。因此，我們可以創建兩個相同的提示來指示LLM： 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


建立這兩個提示後，我們會使用 `client.responses.create` 將它們發送到大型語言模型。 我們將提示儲存在 `input` 變數中，並將角色指定為 `user`。 這是為了模擬用戶發送給聊天機器人的訊息。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-5-mini"


: 

現在我們可以將兩個請求同時發送到 LLM，並檢視我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



即使提示相同且描述相似，我們也可能得到`Grades`屬性的不同格式。

如果你多次運行上述單元格，格式可能是`3.7`或`3.7 GPA`。

這是因為大型語言模型接受以書面提示形式的非結構化數據，並返回同樣的非結構化數據。我們需要有一個結構化格式，這樣在存儲或使用這些數據時才知道會得到什麼。

通過使用函數調用，我們可以確保收到結構化數據。使用函數調用時，LLM實際上不會調用或執行任何函數。相反，我們為LLM的回應創建一個結構，然後利用這些結構化回應來知道在應用程序中要運行哪個函數。
 


![函數呼叫流程圖](../../../../translated_images/zh-HK/Function-Flow.083875364af4f4bb.webp)


然後我們可以取用從函數返回的結果，並將其傳回給 LLM。LLM 隨後會使用自然語言來回應並解答用戶的查詢。


### 使用函數呼叫的應用案例

<strong>呼叫外部工具</strong>
聊天機器人在回答用戶提問方面表現優異。透過使用函數呼叫，聊天機器人可以利用用戶的訊息來完成特定任務。例如，學生可以請聊天機器人「發送電子郵件給我的導師，說我需要更多這科的協助」。這可以呼叫函數 `send_email(to: string, body: string)`


**建立 API 或資料庫查詢**
用戶可以使用自然語言尋找資訊，系統會將其轉換為格式化的查詢或 API 請求。舉例來說，老師可以請求「完成最後一項作業的學生是誰」，這可以呼叫一個名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函數


<strong>建立結構化資料</strong>
用戶可以將一段文字或 CSV 檔案，利用大型語言模型從中提取重要資訊。例如，學生可以將關於和平協議的維基百科文章轉換成 AI 快閃卡。這可以透過函數 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 來完成


## 2. 建立你的第一個函數調用

建立函數調用的過程包括三個主要步驟：
1. 使用你的函數列表和用戶訊息呼叫 Chat Completions API
2. 讀取模型的回應以執行動作，即執行函數或 API 調用
3. 使用你函數的回應，再次呼叫 Chat Completions API，利用該資訊為用戶建立回應。


![函數呼叫流程](../../../../translated_images/zh-HK/LLM-Flow.3285ed8caf4796d7.webp)


### 函式呼叫的元素

#### 使用者輸入

第一步是建立使用者訊息。這可以透過取得文字輸入的值來動態指派，或者您也可以在此處指派一個值。如果這是您第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（最終使用者）。在函式呼叫中，我們會將其指派為 `user` 並提供一個範例問題。


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函式。

接下來我們會定義一個函式及該函式的參數。我們這裡只會使用一個叫做 `search_courses` 的函式，但你可以建立多個函式。

<strong>重要</strong>：函式會包含在系統訊息內送給 LLM，並且會計算在你可用的 token 數量中。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

函數定義結構有多個層級，每個層級都有其自身的屬性。以下是巢狀結構的細分說明：

**頂層函數屬性：**

`name` - 我們希望被呼叫的函數名稱。 

`description` - 此處為函數運作的說明，重要的是需具體且清晰 

`parameters` - 列出您希望模型在回應中產生的值及格式清單 

**參數物件屬性：**

`type` - 參數物件的資料類型（通常為 "object"）

`properties` - 模型將用於其回應的具體值清單 

**個別參數屬性：**

`name` - 由屬性鍵隱含定義（例如 "role"、"product"、"level"）

`type` - 此特定參數的資料類型（例如 "string"、"number"、"boolean"） 

`description` - 特定參數的描述 

**可選屬性：**

`required` - 一個陣列，列出完成函數呼叫所需的參數 


### 呼叫函數 
定義函數後，我們現在需要在呼叫 Chat Completion API 時包含它。我們透過在請求中加入 `functions` 來達成這點。在這個例子中是 `functions=functions`。 

也可以選擇將 `function_call` 設定為 `auto`。這表示我們會讓大型語言模型（LLM）根據使用者訊息決定應該呼叫哪個函數，而不是自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們看看回應，並看看它是如何格式化的：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函數的名稱被呼叫了，並且從使用者訊息中，LLM 能夠找到符合函數參數的資料。


## 3.將函數調用整合到應用程式中。


在我們測試完來自大型語言模型的格式化回應後，現在我們可以將其整合到應用程式中。

### 管理流程

要將這整合到我們的應用程式中，讓我們採取以下步驟：

首先，讓我們呼叫 OpenAI 服務並將訊息儲存在名為 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函數，用來呼叫 Microsoft Learn API 以獲取課程列表： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們接著會檢查模型是否想要調用一個函數。之後，我們會創建一個可用的函數並將其匹配到所調用的函數。 
接著，我們會取得函數的參數並將其對應到由 LLM 傳來的參數。

最後，我們會附加函數調用訊息以及 `search_courses` 訊息返回的值。這會提供 LLM 所需的所有資訊，讓它能使用自然語言
回應用戶。 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將發送更新後的消息到 LLM，以便我們能收到自然語言的回應，而不是 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 程式挑戰 

做得好！要繼續學習 OpenAI 函數呼叫，你可以構建：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 函數的更多參數，可能幫助學習者找到更多課程。你可以在此處找到可用的 API 參數： 
 - 創建另一個函數呼叫，從學習者那裡獲取更多資訊，例如他們的母語 
 - 當函數呼叫和/或 API 呼叫沒有返回任何合適課程時，創建錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
